In [2]:
import pandas as pd
import os
import shutil
import subprocess
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import re
import json

## Generate the antibody specific training data 

In [3]:
input_file = "/home/yujieq/work/ML_training/bNAb-ReP/original_data/env_neu_unique_ab_removed_outliers_duplicates_geomean_include_TBDs.txt"

output_dirs = {
    "50": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining_include_TBDs_5folds_nestedcv/IC50_50",
    "1": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining_include_TBDs_5folds_nestedcv/IC50_1",
    "0.2": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining_include_TBDs_5folds_nestedcv/IC50_0.2"
}

for key in output_dirs:
    os.makedirs(output_dirs[key], exist_ok=True)

In [4]:
r_script_path = "/home/yujieq/work/ML_training/bNAb-ReP/scripts/bNAb-ReP_preprocess_v.1.1-1.R"

mafft_path = os.path.expanduser("~/.conda/envs/deep_learning/bin/mafft")

In [5]:
antibodies_to_include = ['PGT121', 'VRC01', '10-1074', 'PGT145', '3BNC117', 'PGDM1400', 'VRC26.25', 'PGT151', 'PG9', '4E10', 'PGT128', 'SF12', 'N6', '35O22', 'PGT135', 'VRC-PG04', 'CH01', 'HJ16', '10E8', 'VRC34.01', 'b12', 'VRC07-523LS.v34', 'VRC03', 'VRC07', '2F5', '8ANC195']


print(f"Number of antibodies in the list: {len(antibodies_to_include)}")

Number of antibodies in the list: 26


In [6]:
df = pd.read_csv(input_file, sep="\t")

filtered_df = df[df['Antibody'].isin(antibodies_to_include)]

filtered_df

,Antibody,epitope,Virus,clade,IC50,IC80,envseq,heavy,light,lineage
698,10-1074,GP120_V3,001428_2_42,C,0.02436,0.168800,MRVRGILR-NY-QQWW--------MWGVL---GFWMLM--ICNGVE...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
699,10-1074,GP120_V3,0041_V3_C18,C,50.00000,50.000000,MRVRGILR-NW-QLWW--------TWGIL---GFWMVM--NCNVRG...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
700,10-1074,GP120_V3,0077_V1_C16,C,50.00000,50.000000,MRVMGSMR-NC-QRWW--------IWGIL---GFWMLM--TCNMEE...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
701,10-1074,GP120_V3,00836_2_5,C,50.00000,50.000000,MRVRGIRR-NY-QHWW--------IWGIL---GFWMLM--ICKGGR...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
702,10-1074,GP120_V3,0260_V5_C36,A1,0.14914,0.505000,MRVMGIQR-NS-QCFL--------SWGML---VLGIMM--ICSAVG...,QVQLQESGPGLVKPSETLSVTCS------VSG---DSMN-N-----...,SYV----R-PLSVALGETARISCGRQA--L-----G-S-------R...,PGT121
...,...,...,...,...,...,...,...,...,...,...
60639,b12,GP120_CD4BS,ZM233_6,C,50.00000,50.000000,MRVRGIMR-NW-QQWW--------IWGSL---GFWMLI--ICNVMG...,---LEQSGAEVKKPGASVKVSCQ------ASG---YRFS-N-----...,E--LTQAPGTLSLSPGERATFSCRSSHSI------R-S-------R...,unknown
60640,b12,GP120_CD4BS,ZM246F,C,20.08764,NaN,MRVMGILR-NC-QQWW--------IWSIL---GF--LM--IYSVIG...,---LEQSGAEVKKPGASVKVSCQ------ASG---YRFS-N-----...,E--LTQAPGTLSLSPGERATFSCRSSHSI------R-S-------R...,unknown
60641,b12,GP120_CD4BS,ZM247_V1,C,50.00000,NaN,MRAMGIKR-NC-QRWL--------IWGIL---GFWVLL--TYNVMG...,---LEQSGAEVKKPGASVKVSCQ------ASG---YRFS-N-----...,E--LTQAPGTLSLSPGERATFSCRSSHSI------R-S-------R...,unknown
60642,b12,GP120_CD4BS,ZM249_1,C,3.29953,15.933325,MRVMGILR-NC-QPWW--------IWSIL---GFWMLM--BCSG--...,---LEQSGAEVKKPGASVKVSCQ------ASG---YRFS-N-----...,E--LTQAPGTLSLSPGERATFSCRSSHSI------R-S-------R...,unknown


In [7]:
# First filter out rows with empty/missing IC50 values
filtered_df = filtered_df.dropna(subset=['IC50'])

for antibody, group in filtered_df.groupby("Antibody"):
    print(f"Antibody: {antibody}, Antibody instances: {len(group)}")

Antibody: 10-1074, Antibody instances: 1027
Antibody: 10E8, Antibody instances: 829
Antibody: 2F5, Antibody instances: 951
Antibody: 35O22, Antibody instances: 324
Antibody: 3BNC117, Antibody instances: 1086
Antibody: 4E10, Antibody instances: 959
Antibody: 8ANC195, Antibody instances: 279
Antibody: CH01, Antibody instances: 412
Antibody: HJ16, Antibody instances: 328
Antibody: N6, Antibody instances: 530
Antibody: PG9, Antibody instances: 915
Antibody: PGDM1400, Antibody instances: 1171
Antibody: PGT121, Antibody instances: 1327
Antibody: PGT128, Antibody instances: 621
Antibody: PGT135, Antibody instances: 375
Antibody: PGT145, Antibody instances: 616
Antibody: PGT151, Antibody instances: 408
Antibody: SF12, Antibody instances: 138
Antibody: VRC-PG04, Antibody instances: 321
Antibody: VRC01, Antibody instances: 1440
Antibody: VRC03, Antibody instances: 375
Antibody: VRC07, Antibody instances: 400
Antibody: VRC07-523LS.v34, Antibody instances: 205
Antibody: VRC26.25, Antibody instance

In [10]:
def clean_sequence(seq):
    # Replace any non-letter characters with dashes
    cleaned_seq = re.sub(r'[^A-Za-z]', '-', seq)
    # Replace 'B' with 'N'
    cleaned_seq = cleaned_seq.replace('B', 'N')
    # Replace '*' with '-'
    cleaned_seq = cleaned_seq.replace('*', '-')
    # Replace '#' with 'N'
    cleaned_seq = cleaned_seq.replace('#', 'N')
    return cleaned_seq

def process_and_save_files(filtered_df, threshold, threshold_label, mafft_path):
    for antibody, group in filtered_df.groupby("Antibody"):
        print(f"Processing antibody: {antibody}, Instances: {len(group)}")
        
        sanitized_antibody = antibody.replace("/", "_")

        antibody_dir = os.path.join(output_dirs[threshold_label], sanitized_antibody)
        os.makedirs(antibody_dir, exist_ok=True)
        
        sequences = []
        neutralization = []

        for _, row in group.iterrows():
            antibody_str = str(row['Antibody']).replace(" ", "")
            epitope = str(row['epitope']).replace(" ", "")
            virus = row['Virus'].replace(" ", "")
            seq_id = f"{antibody_str}_{epitope}_{virus}"

            envseq_cleaned = clean_sequence(row['envseq'])

            record = SeqRecord(Seq(envseq_cleaned), id=seq_id, description="")
            sequences.append(record)

            ic50 = float(row['IC50'])
            neut_value = 0 if ic50 < threshold else 1
            neutralization.append(str(neut_value))
            
        alignment_filename = f"{sanitized_antibody}_IC50_{threshold_label}_antibody_alignment.fasta"
        neutralization_filename = f"{sanitized_antibody}_IC50_{threshold_label}_antibody_neutralization.txt"
        alignment_file = os.path.join(antibody_dir, alignment_filename)
        neutralization_file = os.path.join(antibody_dir, neutralization_filename)

        with open(alignment_file, "w") as f:
            SeqIO.write(sequences, f, "fasta")

        with open(neutralization_file, "w") as f:
            f.write("\n".join(neutralization))

        shutil.copy(r_script_path, antibody_dir)

        command = f"Rscript {os.path.basename(r_script_path)} {alignment_filename} {neutralization_filename} {mafft_path}"
        subprocess.run(command, shell=True, cwd=antibody_dir)

In [12]:
# Process and save files for different thresholds
process_and_save_files(filtered_df, 1, "1", mafft_path)

Processing antibody: 561_01_18, Instances: 397


Read 397 items
nadd = 1
rescale = 1
dndpre (aa) Version 7.505
alg=X, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

rescale = 1
All-to-all alignment.
  396 / 397

##### writing hat3
pairlocalalign (aa) Version 7.505
alg=Y, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

nadd = 1
nthread = 0
blosum 62 / kimura 200
sueff_global = 0.100000
norg = 397
njobc = 398
Loading 'hat3' ... 
done.
rescale = 1
Loading 'hat2n' (aligned sequences - new sequences) ... done.
done.ng 'hat2i' (aligned sequences) ... 
cTEP 0 / 1                    

Combining ..
   done.                      

   done.                      

addsingle (aa) Version 7.505
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)


To keep the alignment length, 17 letters were DELETED.
To know the positions of deleted letters, rerun the same command with the --mapout option.

Strategy:
 Multi-INS-full (Not tested.)
 ?

If unsure which option to use, try 'mafft --auto input

[1] "Perform one-hot encoding using AA21 (20 AA, glycan)"
[1] "Script duration: 0.27 min"
Processing antibody: M1214_N1, Instances: 120


Read 120 items
nadd = 1
rescale = 1
dndpre (aa) Version 7.505
alg=X, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

rescale = 1
All-to-all alignment.
  119 / 120

##### writing hat3
pairlocalalign (aa) Version 7.505
alg=Y, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

nadd = 1
nthread = 0
blosum 62 / kimura 200
sueff_global = 0.100000
norg = 120
njobc = 121
Loading 'hat3' ... 
done.
rescale = 1
Loading 'hat2n' (aligned sequences - new sequences) ... done.
Loading 'hat2i' (aligned sequences) ... done.
cTEP 0 / 1                    

Combining ..
   done.                      

   done.                      

addsingle (aa) Version 7.505
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)


To keep the alignment length, 8 letters were DELETED.
To know the positions of deleted letters, rerun the same command with the --mapout option.

Strategy:
 Multi-INS-full (Not tested.)
 ?

If unsure which option to use, try 'mafft --auto i

[1] "Perform one-hot encoding using AA21 (20 AA, glycan)"
[1] "Script duration: 0.09 min"


## CV Results

In [1]:
import pandas as pd
import os

In [2]:
output_dirs = {
    "50": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining_include_TBDs_5folds_nestedcv/IC50_50",
    "1": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining_include_TBDs_5folds_nestedcv/IC50_1",
    "0.2": "/home/yujieq/work/ML_training/bNAb-ReP/removed_outliers_duplicates_no_pretraining_include_TBDs_5folds_nestedcv/IC50_0.2"
}
antibodies_to_include = ['PGT121', 'VRC01', '10-1074', 'PGT145', '3BNC117', 'PGDM1400', 'VRC26.25', 'PGT151', 'PG9', '4E10', 'PGT128', 'SF12', 'N6', '35O22', 'PGT135', 'VRC-PG04', 'CH01', 'HJ16', '10E8', 'VRC34.01', 'b12', 'VRC07-523LS.v34', 'VRC03', 'VRC07', '2F5', '8ANC195']

In [3]:
def check_all_files_exist(antibody_dir, num_files=10):
    for i in range(1, num_files + 1):
        file_path = os.path.join(antibody_dir, f"retrain_{i}.csv")
        if not os.path.exists(file_path):
            return False
    return True

def load_and_aggregate_metrics(antibody_dir, antibody, metrics, num_files=10):
    data = {metric: [] for metric in metrics}
    
    for i in range(1, num_files + 1):
        file_path = os.path.join(antibody_dir, f"retrain_{i}.csv")
        
        df = pd.read_csv(file_path)
        df.rename(columns={'Unnamed: 0': 'Metric'}, inplace=True)

        for metric in metrics:
            if metric in df['Metric'].values:  
                metric_row = df[df['Metric'] == metric]
                mean_value = metric_row['mean'].values[0]
                sd_value = metric_row['sd'].values[0]
                data[metric].append((mean_value, sd_value))

    aggregated_data = {}
    for metric in metrics:
        if data[metric]:
            mean_values = [x[0] for x in data[metric]]
            sd_values = [x[1] for x in data[metric]]
            mean_of_means = sum(mean_values) / len(mean_values)
            mean_of_sds = sum(sd_values) / len(sd_values)
            aggregated_data[metric] = (mean_of_means, mean_of_sds)
        else:
            aggregated_data[metric] = (None, None)
    
    return aggregated_data

def process_all_antibodies(output_dirs, antibodies, metrics):
    results = {threshold: {} for threshold in output_dirs}
    
    for threshold_label, directory in output_dirs.items():
        for antibody in antibodies:
            sanitized_antibody = antibody.replace("/", "_")
            antibody_dir = os.path.join(directory, sanitized_antibody)
            if os.path.exists(antibody_dir) and check_all_files_exist(antibody_dir):
                aggregated_data = load_and_aggregate_metrics(antibody_dir, antibody, metrics)
                if aggregated_data is not None:
                    if threshold_label not in results:
                        results[threshold_label] = {}
                    results[threshold_label][sanitized_antibody] = aggregated_data
    
    return results

In [6]:
performance_metrics = ['accuracy', 'auc', 'mcc']

aggregated_results = process_all_antibodies(output_dirs, antibodies_to_include, performance_metrics)


for threshold_label, threshold_data in aggregated_results.items():
    formatted_results = {metric: [] for metric in performance_metrics}
    for antibody, metrics_data in threshold_data.items():
        for metric in performance_metrics:
            if metrics_data[metric]:
                values = metrics_data[metric]
                if values[0] is not None and values[1] is not None:
                    formatted_results[metric].append(f"{values[0]:.2f} ({values[1]:.2f})")
                else:
                    formatted_results[metric].append("N/A")
            else:
                formatted_results[metric].append("N/A")

    df = pd.DataFrame(formatted_results, index=list(threshold_data.keys()))
    df = df.transpose()

    csv_filename = f"threshold_{threshold_label}.csv"
    df.to_csv(csv_filename, index=True)

    print(f"Threshold: {threshold_label}")
    print(df)
    print("\n")

Threshold: 50
            561_01_18           N6       PGT121        VRC01         SF12  \
accuracy  0.81 (0.19)  0.61 (0.29)  0.90 (0.02)  0.93 (0.01)  0.85 (0.08)   
auc       0.76 (0.16)  0.68 (0.22)  0.94 (0.02)  0.89 (0.03)  0.86 (0.08)   
mcc       0.57 (0.19)  0.35 (0.23)  0.78 (0.04)  0.69 (0.05)  0.72 (0.12)   

              10-1074       PGT145     M1214_N1      3BNC117     PGDM1400  \
accuracy  0.96 (0.01)  0.82 (0.04)  0.74 (0.11)  0.93 (0.02)  0.90 (0.02)   
auc       0.97 (0.01)  0.85 (0.04)  0.69 (0.12)  0.91 (0.04)  0.92 (0.02)   
mcc       0.92 (0.02)  0.62 (0.06)  0.57 (0.15)  0.71 (0.07)  0.76 (0.04)   

          ...       PGT151         CH01      DH270.6         4E10  \
accuracy  ...  0.86 (0.04)  0.80 (0.04)  0.92 (0.03)  0.91 (0.04)   
auc       ...  0.88 (0.03)  0.82 (0.05)  0.93 (0.04)  0.78 (0.06)   
mcc       ...  0.71 (0.07)  0.61 (0.08)  0.85 (0.07)  0.50 (0.10)   

               PGT128         HJ16     VRC26.25     VRC-CH31         10E8  \
accuracy  0.88